** Step 1:**  
Imports Spark functions and types, key, sets up storage account and container variables for data processing.

In [0]:
%run "./storage_key"

In [0]:
import requests
import json
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, LongType, BooleanType
# Configurations, path and storage key
storage_account_name = "konstyantyn"
container_name = "konstyantyn"
#path
wiki_url = "https://stream.wikimedia.org/v2/stream/recentchange"
raw_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/raw_wiki_json/"
checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoints/wiki/"
schema_location = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoints/wiki_schema/"

print("Step 1 done!")


** Step 2:**  
Wiki stream data gathering from the URL.

In [0]:

try:
    print("Fetching live data from Wikipedia API...")
    events = []
    
    # Send GET request with custom User-Agent (Wikimedia API requirement)
    headers = {'User-Agent': 'DatabricksLab/1.0 (kostia6@softserve.academy)'}
    
    r = requests.get(wiki_url, stream=True, headers=headers, timeout=15)
    for line in r.iter_lines():
        if line:
            line_str = line.decode('utf-8').strip()
            
            # Strip leading 'data:' if present
            if line_str.startswith('data:'):
                line_str = line_str[5:].strip()
            
            # Try parsing as JSON if it looks like a JSON object
            if line_str.startswith('{') and line_str.endswith('}'):
                try:
                    parsed_event = json.loads(line_str)
                    events.append(parsed_event)
                    print(f"Captured {len(events)}/20: {parsed_event.get('title', 'No title')}")
                    if len(events) >= 20:
                        break
                except json.JSONDecodeError:
                    continue
    r.close()

    # Verify that dataset is not empty
    if events:
        events_df = spark.createDataFrame(events, schema=wiki_schema)
        
        (events_df.write
            .mode("append")
            .json(raw_path)
        )
        print("Step 2 done!")
    else:
        print("No events.")

except Exception as e:
    print(f"Error: {e}")

** Step 3:**  
Wiki stream data reading with Auto Loader and data saving.

In [0]:
try:
    # reading data with schema created in step 2
    stream_df = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.rescuedDataColumn", "_rescued_data")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.schemaHints", """
            id LONG,
            type STRING,
            title STRING,
            namespace LONG,
            comment STRING,
            user STRING,
            bot BOOLEAN,
            minor BOOLEAN,
            server_name STRING,
            timestamp LONG
        """)
        .load(raw_path)
        .withColumn("ingested_at", current_timestamp())
    )




    # Saving data to table
    (stream_df.writeStream
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable("dbr_dev.konstyantyn_bronze.wiki_bronze_data")
        .awaitTermination()
    )
    print("Step 3 done!")
except Exception as e:
    print(f"Error: {e}")

** Step 4:**  
Data checking.

In [0]:
#data test
display(spark.sql("SELECT COUNT(*) FROM dbr_dev.konstyantyn_bronze.wiki_bronze_data"))
display(spark.sql("SELECT * FROM dbr_dev.konstyantyn_bronze.wiki_bronze_data LIMIT 5"))
print("Step 4 done!")

In [0]:

display(spark.sql("""
    SELECT server_name, bot, COUNT(*) as count 
    FROM dbr_dev.konstyantyn_bronze.wiki_bronze_data 
    GROUP BY server_name, bot 
    ORDER BY count DESC
"""))